In [56]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [57]:
import polars as pl
import polars.selectors as cs
from datetime import datetime
from src.commons import data_loader

## -- Configuración profesional de visualización en Polars

In [58]:
# Configuración profesional de visualización en Polars

# Limit the maximum rows and columns shown in prints
pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_cols(15)

pl.Config.set_fmt_str_lengths(300)   # Trunca textos largos a 15 caracteres para no romper la línea
pl.Config.set_tbl_width_chars(150)  # Ajusta el ancho total de la tabla en pantalla


polars.config.Config

## -- Constants Variables for File names

In [55]:
UTILITY_PROVIDER_FILE = "utility_provider.csv"
SUBSTATION_FILE = "substation.csv"
CONSUMER_FILE = "consumer.csv"
AMI_HEAD_END_FILE = "ami_head_end.csv"
DATA_MGMT_SYSTEM_FILE = "data_mgmt_system.csv"
DISTRIBUTION_NETWORK_FILE = "distribution_network.csv"
DISTRIBUTION_TRANSFORMER_FILE = "distribution_transformer.csv"
ENERGY_STORAGE_FILE = "energy_storage.csv"
POWER_PLANT_FILE = "power_plants.csv"
POWER_TRANSFORMER_FILE = "power_transformer.csv"
RENEWABLE_SOURCE_FILE = "renewable_source.csv"
SCADA_DMS_FILE = "scada_dms.csv"
SMART_METERS_FILE = "smart_meters.csv"

## -- Data Ingestion from CSV files

In [79]:
df_utility_provider = data_loader.read_csv(UTILITY_PROVIDER_FILE)

In [107]:
df_substation = data_loader.read_csv(SUBSTATION_FILE)

In [8]:
df_consumer = data_loader.read_csv(CONSUMER_FILE)

In [12]:
df_ami_head_end = data_loader.read_csv(AMI_HEAD_END_FILE)

In [16]:
df_data_mgmt_system = data_loader.read_csv(DATA_MGMT_SYSTEM_FILE)

In [22]:
df_distribution_network = data_loader.read_csv(DISTRIBUTION_NETWORK_FILE)

In [26]:
df_distribution_transformer = data_loader.read_csv(DISTRIBUTION_TRANSFORMER_FILE)

In [32]:
df_energy_storage = data_loader.read_csv(ENERGY_STORAGE_FILE)

In [38]:
df_power_plants =  data_loader.read_csv(POWER_PLANT_FILE)

In [41]:
df_power_transformer = data_loader.read_csv(POWER_TRANSFORMER_FILE)

In [45]:
df_renewable_source = data_loader.read_csv(RENEWABLE_SOURCE_FILE)

In [49]:
df_scada_dms = data_loader.read_csv(SCADA_DMS_FILE)

In [59]:
df_smart_meters = data_loader.read_csv(SMART_METERS_FILE)

## -- DATA CLEANSING & HARMONIZATION - UTILITY PROVIDER

In [100]:
# 1 Pipeline Execution
cleaned_utility_provider_df = (
    df_utility_provider
    # --- STEP 1: NAMING CONVENTIONS ---
    # Convert all columns to lowercase and replace spaces/hyphens with underscores
    .rename({col: col.lower().strip().replace(" ", "_") for col in df_utility_provider.columns})
    
  
    # --- STEP 2: DELETE EMPTY ROWS OR WHITE LINES ---
    # If ID of the substation is null, it is empty or white space, it removes the complete row.
    .filter(
        pl.col("provider_id").is_not_null() & 
        (pl.col("provider_id").str.strip_chars() != "")
    )
    # STEP 3: ADD INGESTION META DATA
    .with_columns([
        # Text alignment: trim whitespace and capitalize names cleanly
        pl.col("name").str.strip_chars(),
        pl.col("region").str.strip_chars(),
        # Adding ingestion metada
        pl.lit(datetime.now()).alias("ingested_at"), 
        pl.lit(UTILITY_PROVIDER_FILE).alias("ingested_from_file"),
    ])
    
    # --- STEP 4: TEXT HARMONIZATION ---
    # It will only apply "UNKNOWN" to empty cells actually empty from the valid rows
    .with_columns([
        pl.when(
            pl.col(col).is_null() | (pl.col(col).str.strip_chars() == "")
        )
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col(col).str.strip_chars())
        .alias(col)
        for col in ["name", "region"]
    ])
    
    # --- STEP 5: DEDUPLICATION ---
    # Drop exact duplicates, keeping the first occurrence
    .unique(maintain_order=True)
    
    # Drop rows where critical identifying keys are missing completely
    .filter(pl.col("provider_id").is_not_null())
)

print(cleaned_utility_provider_df)


shape: (5, 5)
┌─────────────┬─────────────────────────┬─────────┬─────────────────┬──────────────────────┐
│ provider_id ┆ name                    ┆ region  ┆ ingested_at     ┆ ingested_from_file   │
│ ---         ┆ ---                     ┆ ---     ┆ ---             ┆ ---                  │
│ str         ┆ str                     ┆ str     ┆ datetime[μs]    ┆ str                  │
╞═════════════╪═════════════════════════╪═════════╪═════════════════╪══════════════════════╡
│ UP-001      ┆ Andes Power Co          ┆ North   ┆ 2026-08-28      ┆ utility_provider.csv │
│             ┆                         ┆         ┆ 04:44:58.346682 ┆                      │
│ UP-002      ┆ Pacifica Energy         ┆ South   ┆ 2026-08-28      ┆ utility_provider.csv │
│             ┆                         ┆         ┆ 04:44:58.346682 ┆                      │
│ UP-003      ┆ Northern Grid Utilities ┆ East    ┆ 2026-08-28      ┆ utility_provider.csv │
│             ┆                         ┆         ┆ 04:4

## -- DATA CLEANSING & HARMONIZATION   - SUBSTATION

In [112]:
# 1 Execution Pipeline
    
# 2. Pipeline cleansing data
cleaned_substation_df = (
    df_substation
    
    # --- STEP 1: NAMING CONVENSION ---
    .rename({col: col.strip().lower().replace(" ", "_") for col in df_substation.columns})
    
    # --- STEP 2: DROP EMPTY OR NULL ROWS ---
    # If the ID of the substation is null, or is empty, it dropss the complete row
    .filter(
        pl.col("substation_id").is_not_null() & 
        (pl.col("substation_id").str.strip_chars() != "")
    )
    .with_columns([
        pl.col("voltage_kv").fill_null(pl.col("voltage_kv").median()).cast(target_dtype)
    ])
    
    # --- STEP 3: ADD INGESTION META DATA ---
    .with_columns([
        pl.lit(datetime.now()).alias("ingested_at"), 
        pl.lit(SUBSTATION_FILE).alias("ingested_from_file"),
    ])
    
    # --- PASO 4: TEXT HARMONIZATION ---
    # Now it will only apply UNKNOWN" to empy valid rows
    .with_columns([
        pl.when(
            pl.col(col).is_null() | (pl.col(col).str.strip_chars() == "")
        )
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col(col).str.strip_chars())
        .alias(col)
        for col in ["substation_type", "location", "source_type", "source_id", "scada_id"]
    ])
    
    # --- PASO 5: DEDUPLICATION ---
    .unique(maintain_order=True)
)

# 3. Show trust results
print(cleaned_substation_df)

shape: (500, 9)
┌───────────────┬─────────────────┬──────────────┬────────────┬───┬───────────┬──────────┬────────────────────────────┬────────────────────┐
│ substation_id ┆ substation_type ┆ location     ┆ voltage_kv ┆ … ┆ source_id ┆ scada_id ┆ ingested_at                ┆ ingested_from_file │
│ ---           ┆ ---             ┆ ---          ┆ ---        ┆   ┆ ---       ┆ ---      ┆ ---                        ┆ ---                │
│ str           ┆ str             ┆ str          ┆ f64        ┆   ┆ str       ┆ str      ┆ datetime[μs]               ┆ str                │
╞═══════════════╪═════════════════╪══════════════╪════════════╪═══╪═══════════╪══════════╪════════════════════════════╪════════════════════╡
│ SUB-0001      ┆ Step-down       ┆ District 80  ┆ 138.0      ┆ … ┆ PP-0004   ┆ SCD-003  ┆ 2026-08-28 04:47:54.246738 ┆ substation.csv     │
│ SUB-0002      ┆ Step-down       ┆ District 119 ┆ 138.0      ┆ … ┆ RS-0024   ┆ SCD-001  ┆ 2026-08-28 04:47:54.246738 ┆ substation.csv    

## -- DATA CLEANSING & HARMONIZATION   - CONSUMER

In [10]:
# 1 Execution Pipeline
    
# 2. Cleansing with polars
cleaned_consumer_df = (
    df_consumer
    
    # --- STEP 1: NAMING CONVENSION ---
    .rename({col: col.strip().lower().replace(" ", "_") for col in df_consumer.columns})
    
    # --- STEP 2: DROP EMPTY OR NULL ROWS---
    # if the ID of the consumer is null, or it is blank or white space, it drops the whole row
    .filter(
        pl.col("consumer_id").is_not_null() & 
        (pl.col("consumer_id").str.strip_chars() != "")
    )
    
    # --- STEP 3: INGESTION META DATA  ---
    .with_columns([
        pl.lit(datetime.now()).alias("ingested_at"), 
        pl.lit(CONSUMER_FILE).alias("ingested_from_file"),
    ])
    
    # --- STEP 4: TEXT HARMONIZATION ---
    # Not it will only apply "UNKNOWN" to empty valid rows
    .with_columns([
        pl.when(
            pl.col(col).is_null() | (pl.col(col).str.strip_chars() == "")
        )
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col(col).str.strip_chars())
        .alias(col)
        for col in ["account_type", "address"]
    ])
    
    # --- STEP 5: DEDUPLICATION ---
    .unique(maintain_order=True)
)

# 3. Show thrust result
print(cleaned_consumer_df.head(20))

shape: (20, 5)
┌──────────────┬──────────────┬───────────────────────────────┬────────────────────────────┬────────────────────┐
│ consumer_id  ┆ account_type ┆ address                       ┆ ingested_at                ┆ ingested_from_file │
│ ---          ┆ ---          ┆ ---                           ┆ ---                        ┆ ---                │
│ str          ┆ str          ┆ str                           ┆ datetime[μs]               ┆ str                │
╞══════════════╪══════════════╪═══════════════════════════════╪════════════════════════════╪════════════════════╡
│ CON-00000001 ┆ Residential  ┆ 6818 Hill St, Stonebridge     ┆ 2026-08-28 04:56:29.818190 ┆ consumer.csv       │
│ CON-00000002 ┆ Commercial   ┆ 399 Lake Dr, Clearwater       ┆ 2026-08-28 04:56:29.818190 ┆ consumer.csv       │
│ CON-00000003 ┆ Residential  ┆ 6673 Hill St, Lakeside        ┆ 2026-08-28 04:56:29.818190 ┆ consumer.csv       │
│ CON-00000004 ┆ Residential  ┆ 5824 Main St, Fairview        ┆ 2026-08-2

## -- DATA CLEANSING & HARMONIZATION   - AMI HEAD END

In [14]:
# 1 Execution Pipeline
    
# 2. Cleansing with polars
cleaned_ami_head_end = (
    df_ami_head_end
    
    # --- STEP 1: NAMING CONVENSION ---
    .rename({col: col.strip().lower().replace(" ", "_") for col in df_ami_head_end.columns})
    
    # --- STEP 2: DROP EMPTY OR NULL ROWS---
    # if the ID of the ami_head_end is null, or it is blank or white space, it drops the whole row
    .filter(
        pl.col("hes_id").is_not_null() & 
        (pl.col("hes_id").str.strip_chars() != "")
    )
    
    # --- STEP 3: INGESTION META DATA  ---
    .with_columns([
        pl.lit(datetime.now()).alias("ingested_at"), 
        pl.lit(AMI_HEAD_END_FILE).alias("ingested_from_file"),
    ])
    
    # --- STEP 4: TEXT HARMONIZATION ---
    # Not it will only apply "UNKNOWN" to empty valid rows
    .with_columns([
        pl.when(
            pl.col(col).is_null() | (pl.col(col).str.strip_chars() == "")
        )
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col(col).str.strip_chars())
        .alias(col)
        for col in ["network_type", "coverage_area", "dms_system"]
    ])
    
    # --- STEP 5: DEDUPLICATION ---
    .unique(maintain_order=True)
)

# 3. Show thrust result
print(cleaned_ami_head_end)

shape: (200, 6)
┌─────────┬──────────────┬───────────────┬───────────────┬────────────────────────────┬────────────────────┐
│ hes_id  ┆ network_type ┆ coverage_area ┆ dms_system_id ┆ ingested_at                ┆ ingested_from_file │
│ ---     ┆ ---          ┆ ---           ┆ ---           ┆ ---                        ┆ ---                │
│ str     ┆ str          ┆ str           ┆ str           ┆ datetime[μs]               ┆ str                │
╞═════════╪══════════════╪═══════════════╪═══════════════╪════════════════════════════╪════════════════════╡
│ HES-001 ┆ RF-Mesh      ┆ Zone 71       ┆ DMS-004       ┆ 2026-08-28 05:05:31.826232 ┆ ami_head_end.csv   │
│ HES-002 ┆ RF-Mesh      ┆ Zone 55       ┆ DMS-003       ┆ 2026-08-28 05:05:31.826232 ┆ ami_head_end.csv   │
│ HES-003 ┆ Cellular     ┆ Zone 48       ┆ DMS-004       ┆ 2026-08-28 05:05:31.826232 ┆ ami_head_end.csv   │
│ HES-004 ┆ RF-Mesh      ┆ Zone 17       ┆ DMS-002       ┆ 2026-08-28 05:05:31.826232 ┆ ami_head_end.csv   │
│ H

## -- DATA CLEANSING & HARMONIZATION   - DATA MANAGEMENT SYSTEM

In [20]:
# 1 Execution Pipeline
    
# 2. Cleansing with polars
cleaned_data_mgmt_system_df = (
    df_data_mgmt_system
    
    # --- STEP 1: NAMING CONVENSION ---
    .rename({col: col.strip().lower().replace(" ", "_") for col in df_data_mgmt_system.columns})
    
    # --- STEP 2: DROP EMPTY OR NULL ROWS---
    # if the ID of the ami_head_end is null, or it is blank or white space, it drops the whole row
    .filter(
        pl.col("system_id").is_not_null() & 
        (pl.col("system_id").str.strip_chars() != "")
    )
    
    # --- STEP 3: INGESTION META DATA  ---
    .with_columns([
        pl.lit(datetime.now()).alias("ingested_at"), 
        pl.lit(DATA_MGMT_SYSTEM_FILE).alias("ingested_from_file"),
    ])
    
    # --- STEP 4: TEXT HARMONIZATION ---
    # Not it will only apply "UNKNOWN" to empty valid rows
    .with_columns([
        pl.when(
            pl.col(col).is_null() | (pl.col(col).str.strip_chars() == "")
        )
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col(col).str.strip_chars())
        .alias(col)
        for col in ["provider_id", "storage_type", "analytics_engine"]
    ])
    
    # --- STEP 5: DEDUPLICATION ---
    .unique(maintain_order=True)
)

# 3. Show thrust result
print(cleaned_data_mgmt_system_df)

shape: (5, 6)
┌───────────┬─────────────┬────────────────┬──────────────────┬────────────────────────────┬──────────────────────┐
│ system_id ┆ provider_id ┆ storage_type   ┆ analytics_engine ┆ ingested_at                ┆ ingested_from_file   │
│ ---       ┆ ---         ┆ ---            ┆ ---              ┆ ---                        ┆ ---                  │
│ str       ┆ str         ┆ str            ┆ str              ┆ datetime[μs]               ┆ str                  │
╞═══════════╪═════════════╪════════════════╪══════════════════╪════════════════════════════╪══════════════════════╡
│ DMS-001   ┆ UP-005      ┆ Data Lake      ┆ DuckDB           ┆ 2026-08-28 05:13:29.826462 ┆ data_mgmt_system.csv │
│ DMS-002   ┆ UP-002      ┆ Data Lake      ┆ Spark            ┆ 2026-08-28 05:13:29.826462 ┆ data_mgmt_system.csv │
│ DMS-003   ┆ UP-004      ┆ Data Warehouse ┆ Presto           ┆ 2026-08-28 05:13:29.826462 ┆ data_mgmt_system.csv │
│ DMS-004   ┆ UP-001      ┆ Hybrid         ┆ Presto       

## -- DATA CLEANSING & HARMONIZATION   - DISTRIBUTION NETWORK

In [24]:
# 1 Execution Pipeline
    
# 2. Cleansing with polars
cleaned_distribution_network_df = (
    df_distribution_network
    
    # --- STEP 1: NAMING CONVENSION ---
    .rename({col: col.strip().lower().replace(" ", "_") for col in df_distribution_network.columns})
    
    # --- STEP 2: DROP EMPTY OR NULL ROWS---
    # if the ID of the ami_head_end is null, or it is blank or white space, it drops the whole row
    .filter(
        pl.col("network_id").is_not_null() & 
        (pl.col("network_id").str.strip_chars() != "")
    )
    
    # --- STEP 3: INGESTION META DATA  ---
    .with_columns([
        pl.lit(datetime.now()).alias("ingested_at"), 
        pl.lit(DISTRIBUTION_NETWORK_FILE).alias("ingested_from_file"),
    ])
    
    # --- STEP 4: TEXT HARMONIZATION ---
    # Not it will only apply "UNKNOWN" to empty valid rows
    .with_columns([
        pl.when(
            pl.col(col).is_null() | (pl.col(col).str.strip_chars() == "")
        )
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col(col).str.strip_chars())
        .alias(col)
        for col in ["substation_id", "feeder_type", "scada_id"]
    ])
    
    # --- STEP 5: DEDUPLICATION ---
    .unique(maintain_order=True)
)

# 3. Show thrust result
print(cleaned_distribution_network_df)

shape: (2_000, 6)
┌────────────┬───────────────┬─────────────┬──────────┬────────────────────────────┬──────────────────────────┐
│ network_id ┆ substation_id ┆ feeder_type ┆ scada_id ┆ ingested_at                ┆ ingested_from_file       │
│ ---        ┆ ---           ┆ ---         ┆ ---      ┆ ---                        ┆ ---                      │
│ str        ┆ str           ┆ str         ┆ str      ┆ datetime[μs]               ┆ str                      │
╞════════════╪═══════════════╪═════════════╪══════════╪════════════════════════════╪══════════════════════════╡
│ DN-00001   ┆ SUB-0362      ┆ Mixed       ┆ SCD-006  ┆ 2026-08-28 06:34:46.797356 ┆ distribution_network.csv │
│ DN-00002   ┆ SUB-0051      ┆ Overhead    ┆ SCD-009  ┆ 2026-08-28 06:34:46.797356 ┆ distribution_network.csv │
│ DN-00003   ┆ SUB-0395      ┆ Underground ┆ SCD-009  ┆ 2026-08-28 06:34:46.797356 ┆ distribution_network.csv │
│ DN-00004   ┆ SUB-0330      ┆ Underground ┆ SCD-003  ┆ 2026-08-28 06:34:46.797356 ┆ d

## -- DATA CLEANSING & HARMONIZATION   - DISTRIBUTION TRANSFORMER

In [27]:
print(df_distribution_transformer.head(5))

shape: (5, 3)
┌────────────────┬────────────┬───────────┐
│ Transformer ID ┆ Network ID ┆ Rated KVA │
│ ---            ┆ ---        ┆ ---       │
│ str            ┆ str        ┆ i64       │
╞════════════════╪════════════╪═══════════╡
│ DXF-000001     ┆ DN-00261   ┆ 75        │
│ DXF-000002     ┆ DN-00424   ┆ 75        │
│ DXF-000003     ┆ DN-00844   ┆ 50        │
│ DXF-000004     ┆ DN-00037   ┆ 50        │
│ DXF-000005     ┆ DN-00791   ┆ 250       │
└────────────────┴────────────┴───────────┘


In [30]:
# 1 Execution Pipeline
    
# 2. Cleansing with polars
cleaned_distribution_transformer_df = (
    df_distribution_transformer
    
    # --- STEP 1: NAMING CONVENSION ---
    .rename({col: col.strip().lower().replace(" ", "_") for col in df_distribution_transformer.columns})
    
    # --- STEP 2: DROP EMPTY OR NULL ROWS---
    # if the ID of the ami_head_end is null, or it is blank or white space, it drops the whole row
    .filter(
        pl.col("transformer_id").is_not_null() & 
        (pl.col("transformer_id").str.strip_chars() != "")
    )

    .with_columns([
        pl.col("rated_kva").fill_null(pl.col("rated_kva").median()).cast(target_dtype)
    ])
    
    # --- STEP 3: INGESTION META DATA  ---
    .with_columns([
        pl.lit(datetime.now()).alias("ingested_at"), 
        pl.lit(DISTRIBUTION_TRANSFORMER_FILE).alias("ingested_from_file"),
    ])
    
    # --- STEP 4: TEXT HARMONIZATION ---
    # Not it will only apply "UNKNOWN" to empty valid rows
    .with_columns([
        pl.when(
            pl.col(col).is_null() | (pl.col(col).str.strip_chars() == "")
        )
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col(col).str.strip_chars())
        .alias(col)
        for col in ["network_id"]
    ])
    
    # --- STEP 5: DEDUPLICATION ---
    .unique(maintain_order=True)
)

# 3. Show thrust result
print(cleaned_distribution_transformer_df)# 1 Execution Pipeline
    
# 2. Cleansing with polars
cleaned_distribution_transformer_df = (
    df_distribution_transformer
    
    # --- STEP 1: NAMING CONVENSION ---
    .rename({col: col.strip().lower().replace(" ", "_") for col in df_distribution_transformer.columns})
    
    # --- STEP 2: DROP EMPTY OR NULL ROWS---
    # if the ID of the ami_head_end is null, or it is blank or white space, it drops the whole row
    .filter(
        pl.col("transformer_id").is_not_null() & 
        (pl.col("transformer_id").str.strip_chars() != "")
    )

    .with_columns([
        pl.col("rated_kva").fill_null(pl.col("rated_kva").median()).cast(target_dtype)
    ])
    
    # --- STEP 3: INGESTION META DATA  ---
    .with_columns([
        pl.lit(datetime.now()).alias("ingested_at"), 
        pl.lit(DISTRIBUTION_TRANSFORMER_FILE).alias("ingested_from_file"),
    ])
    
    # --- STEP 4: TEXT HARMONIZATION ---
    # Not it will only apply "UNKNOWN" to empty valid rows
    .with_columns([
        pl.when(
            pl.col(col).is_null() | (pl.col(col).str.strip_chars() == "")
        )
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col(col).str.strip_chars())
        .alias(col)
        for col in ["network_id"]
    ])
    
    # --- STEP 5: DEDUPLICATION ---
    .unique(maintain_order=True)
)

# 3. Show thrust result
print(cleaned_distribution_transformer_df)

shape: (150_000, 5)
┌────────────────┬────────────┬───────────┬────────────────────────────┬──────────────────────────────┐
│ transformer_id ┆ network_id ┆ rated_kva ┆ ingested_at                ┆ ingested_from_file           │
│ ---            ┆ ---        ┆ ---       ┆ ---                        ┆ ---                          │
│ str            ┆ str        ┆ f64       ┆ datetime[μs]               ┆ str                          │
╞════════════════╪════════════╪═══════════╪════════════════════════════╪══════════════════════════════╡
│ DXF-000001     ┆ DN-00261   ┆ 75.0      ┆ 2026-08-28 06:59:23.445687 ┆ distribution_transformer.csv │
│ DXF-000002     ┆ DN-00424   ┆ 75.0      ┆ 2026-08-28 06:59:23.445687 ┆ distribution_transformer.csv │
│ DXF-000003     ┆ DN-00844   ┆ 50.0      ┆ 2026-08-28 06:59:23.445687 ┆ distribution_transformer.csv │
│ DXF-000004     ┆ DN-00037   ┆ 50.0      ┆ 2026-08-28 06:59:23.445687 ┆ distribution_transformer.csv │
│ DXF-000005     ┆ DN-00791   ┆ 250.0     ┆ 

## -- DATA CLEANSING & HARMONIZATION   - ENERGY STORAGE

In [37]:
# 1 Execution Pipeline
    
# 2. Cleansing with polars
cleaned_energy_storage_df = (
    df_energy_storage
    
    # --- STEP 1: NAMING CONVENSION ---
    .rename({col: col.strip().lower().replace(" ", "_") for col in df_energy_storage.columns})
    
    # --- STEP 2: DROP EMPTY OR NULL ROWS---
    # if the ID of the ami_head_end is null, or it is blank or white space, it drops the whole row
    .filter(
        pl.col("storage_id").is_not_null() & 
        (pl.col("storage_id").str.strip_chars() != "")
    )

    .with_columns([
        pl.col("capacity_kwh").fill_null(pl.col("capacity_kwh").median())
    ])
    
    # --- STEP 3: INGESTION META DATA  ---
    .with_columns([
        pl.lit(datetime.now()).alias("ingested_at"), 
        pl.lit(ENERGY_STORAGE_FILE).alias("ingested_from_file"),
    ])
    
    # --- STEP 4: TEXT HARMONIZATION ---
    # Not it will only apply "UNKNOWN" to empty valid rows
    .with_columns([
        pl.when(
            pl.col(col).is_null() | (pl.col(col).str.strip_chars() == "")
        )
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col(col).str.strip_chars())
        .alias(col)
        for col in ["type"]
    ])
    
    # --- STEP 5: DEDUPLICATION ---
    .unique(maintain_order=True)
)

# 3. Show thrust result
print(cleaned_energy_storage_df)# 1 Execution Pipeline

shape: (20, 5)
┌────────────┬──────────────┬──────────────┬────────────────────────────┬────────────────────┐
│ storage_id ┆ type         ┆ capacity_kwh ┆ ingested_at                ┆ ingested_from_file │
│ ---        ┆ ---          ┆ ---          ┆ ---                        ┆ ---                │
│ str        ┆ str          ┆ f64          ┆ datetime[μs]               ┆ str                │
╞════════════╪══════════════╪══════════════╪════════════════════════════╪════════════════════╡
│ ES-0001    ┆ Lithium-Ion  ┆ 58825.7      ┆ 2026-08-28 07:33:19.639101 ┆ energy_storage.csv │
│ ES-0002    ┆ Flow Battery ┆ 65334.8      ┆ 2026-08-28 07:33:19.639101 ┆ energy_storage.csv │
│ ES-0003    ┆ Pumped Hydro ┆ 9360.0       ┆ 2026-08-28 07:33:19.639101 ┆ energy_storage.csv │
│ ES-0004    ┆ Lithium-Ion  ┆ 42164.9      ┆ 2026-08-28 07:33:19.639101 ┆ energy_storage.csv │
│ ES-0005    ┆ Pumped Hydro ┆ 5119.8       ┆ 2026-08-28 07:33:19.639101 ┆ energy_storage.csv │
│ ES-0006    ┆ Flow Battery ┆ 49905

## -- DATA CLEANSING & HARMONIZATION   - POWER PLANTS

In [40]:
# 1 Execution Pipeline
    
# 2. Cleansing with polars
cleaned_power_plants_df = (
    df_power_plants
    
    # --- STEP 1: NAMING CONVENSION ---
    .rename({col: col.strip().lower().replace(" ", "_") for col in df_power_plants.columns})
    
    # --- STEP 2: DROP EMPTY OR NULL ROWS---
    # if the ID of the ami_head_end is null, or it is blank or white space, it drops the whole row
    .filter(
        pl.col("plant_id").is_not_null() & 
        (pl.col("plant_id").str.strip_chars() != "")
    )

    .with_columns([
        pl.col("capacity_mw").fill_null(pl.col("capacity_mw").median())
    ])
    
    # --- STEP 3: INGESTION META DATA  ---
    .with_columns([
        pl.lit(datetime.now()).alias("ingested_at"), 
        pl.lit(ENERGY_STORAGE_FILE).alias("ingested_from_file"),
    ])
    
    # --- STEP 4: TEXT HARMONIZATION ---
    # Not it will only apply "UNKNOWN" to empty valid rows
    .with_columns([
        pl.when(
            pl.col(col).is_null() | (pl.col(col).str.strip_chars() == "")
        )
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col(col).str.strip_chars())
        .alias(col)
        for col in ["provider_id","type"]
    ])
    
    # --- STEP 5: DEDUPLICATION ---
    .unique(maintain_order=True)
)

# 3. Show thrust result
print(cleaned_power_plants_df)# 1 Execution Pipeline

shape: (25, 6)
┌──────────┬─────────────┬─────────┬─────────────┬────────────────────────────┬────────────────────┐
│ plant_id ┆ provider_id ┆ type    ┆ capacity_mw ┆ ingested_at                ┆ ingested_from_file │
│ ---      ┆ ---         ┆ ---     ┆ ---         ┆ ---                        ┆ ---                │
│ str      ┆ str         ┆ str     ┆ f64         ┆ datetime[μs]               ┆ str                │
╞══════════╪═════════════╪═════════╪═════════════╪════════════════════════════╪════════════════════╡
│ PP-0001  ┆ UP-001      ┆ Coal    ┆ 332.2       ┆ 2026-08-28 07:38:45.948554 ┆ energy_storage.csv │
│ PP-0002  ┆ UP-004      ┆ Gas     ┆ 726.7       ┆ 2026-08-28 07:38:45.948554 ┆ energy_storage.csv │
│ PP-0003  ┆ UP-004      ┆ Diesel  ┆ 113.5       ┆ 2026-08-28 07:38:45.948554 ┆ energy_storage.csv │
│ PP-0004  ┆ UP-003      ┆ Gas     ┆ 273.7       ┆ 2026-08-28 07:38:45.948554 ┆ energy_storage.csv │
│ PP-0005  ┆ UP-003      ┆ Gas     ┆ 1040.4      ┆ 2026-08-28 07:38:45.94855

## -- DATA CLEANSING & HARMONIZATION   - POWER TRANSFORMER

In [43]:
# 1 Execution Pipeline
    
# 2. Cleansing with polars
cleaned_power_transformer_df = (
    df_power_transformer
    
    # --- STEP 1: NAMING CONVENSION ---
    .rename({col: col.strip().lower().replace(" ", "_") for col in df_power_transformer.columns})
    
    # --- STEP 2: DROP EMPTY OR NULL ROWS---
    # if the ID of the ami_head_end is null, or it is blank or white space, it drops the whole row
    .filter(
        pl.col("transformer_id").is_not_null() & 
        (pl.col("transformer_id").str.strip_chars() != "")
    )

    .with_columns([
        pl.col("capacity_mva").fill_null(pl.col("capacity_mva").median())
    ])
    
    # --- STEP 3: INGESTION META DATA  ---
    .with_columns([
        pl.lit(datetime.now()).alias("ingested_at"), 
        pl.lit(POWER_TRANSFORMER_FILE).alias("ingested_from_file"),
    ])
    
    # --- STEP 4: TEXT HARMONIZATION ---
    # Not it will only apply "UNKNOWN" to empty valid rows
    .with_columns([
        pl.when(
            pl.col(col).is_null() | (pl.col(col).str.strip_chars() == "")
        )
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col(col).str.strip_chars())
        .alias(col)
        for col in ["substation_id"]
    ])
    
    # --- STEP 5: DEDUPLICATION ---
    .unique(maintain_order=True)
)

# 3. Show thrust result
print(cleaned_power_transformer_df)

shape: (1_000, 5)
┌────────────────┬───────────────┬──────────────┬────────────────────────────┬───────────────────────┐
│ transformer_id ┆ substation_id ┆ capacity_mva ┆ ingested_at                ┆ ingested_from_file    │
│ ---            ┆ ---           ┆ ---          ┆ ---                        ┆ ---                   │
│ str            ┆ str           ┆ f64          ┆ datetime[μs]               ┆ str                   │
╞════════════════╪═══════════════╪══════════════╪════════════════════════════╪═══════════════════════╡
│ PXF-00001      ┆ SUB-0314      ┆ 466.7        ┆ 2026-08-28 07:42:49.408217 ┆ power_transformer.csv │
│ PXF-00002      ┆ SUB-0447      ┆ 344.1        ┆ 2026-08-28 07:42:49.408217 ┆ power_transformer.csv │
│ PXF-00003      ┆ SUB-0064      ┆ 487.5        ┆ 2026-08-28 07:42:49.408217 ┆ power_transformer.csv │
│ PXF-00004      ┆ SUB-0006      ┆ 214.3        ┆ 2026-08-28 07:42:49.408217 ┆ power_transformer.csv │
│ PXF-00005      ┆ SUB-0275      ┆ 273.3        ┆ 2026-

## -- DATA CLEANSING & HARMONIZATION   - RENEWABLE SOURCE

In [48]:
# 1 Execution Pipeline
    
# 2. Cleansing with polars
cleaned_renewable_source_df = (
    df_renewable_source
    
    # --- STEP 1: NAMING CONVENSION ---
    .rename({col: col.strip().lower().replace(" ", "_") for col in df_renewable_source.columns})
    
    # --- STEP 2: DROP EMPTY OR NULL ROWS---
    # if the ID of the ami_head_end is null, or it is blank or white space, it drops the whole row
    .filter(
        pl.col("source_id").is_not_null() & 
        (pl.col("source_id").str.strip_chars() != "")
    )

    .with_columns([
        pl.col("output_kw").fill_null(pl.col("output_kw").median())
    ])
    
    # --- STEP 3: INGESTION META DATA  ---
    .with_columns([
        pl.lit(datetime.now()).alias("ingested_at"), 
        pl.lit(RENEWABLE_SOURCE_FILE).alias("ingested_from_file"),
    ])
    
    # --- STEP 4: TEXT HARMONIZATION ---
    # Not it will only apply "UNKNOWN" to empty valid rows
    .with_columns([
        pl.when(
            pl.col(col).is_null() | (pl.col(col).str.strip_chars() == "")
        )
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col(col).str.strip_chars())
        .alias(col)
        for col in ["type"]
    ])
    
    # --- STEP 5: DEDUPLICATION ---
    .unique(maintain_order=True)
)

# 3. Show thrust result
print(df_renewable_source)

shape: (40, 3)
┌───────────┬────────────┬───────────┐
│ Source ID ┆ Type       ┆ Output KW │
│ ---       ┆ ---        ┆ ---       │
│ str       ┆ str        ┆ f64       │
╞═══════════╪════════════╪═══════════╡
│ RS-0001   ┆ Geothermal ┆ 10807.8   │
│ RS-0002   ┆ Solar      ┆ 20485.6   │
│ RS-0003   ┆ Geothermal ┆ 42684.8   │
│ RS-0004   ┆ Solar      ┆ 11773.6   │
│ RS-0005   ┆ Geothermal ┆ 3009.3    │
│ RS-0006   ┆ Geothermal ┆ 14141.1   │
│ RS-0007   ┆ Geothermal ┆ 14750.3   │
│ RS-0008   ┆ Biomass    ┆ 33129.6   │
│ RS-0009   ┆ Wind       ┆ 27895.9   │
│ RS-0010   ┆ Biomass    ┆ 39216.5   │
│ RS-0011   ┆ Wind       ┆ 33249.2   │
│ RS-0012   ┆ Geothermal ┆ 20378.7   │
│ RS-0013   ┆ Biomass    ┆ 40719.6   │
│ RS-0014   ┆ Wind       ┆ 8431.9    │
│ RS-0015   ┆ Biomass    ┆ 1233.3    │
│ RS-0016   ┆ Biomass    ┆ 4593.4    │
│ RS-0017   ┆ Solar      ┆ 36145.7   │
│ RS-0018   ┆ Solar      ┆ 23147.7   │
│ RS-0019   ┆ Solar      ┆ 8147.5    │
│ RS-0020   ┆ Solar      ┆ 25102.1   │
│ RS-0021 

## -- DATA CLEANSING & HARMONIZATION   - SCADA DMS

In [52]:
# 1 Execution Pipeline
    
# 2. Cleansing with polars
cleaned_scada_dms_df = (
    df_scada_dms
    
    # --- STEP 1: NAMING CONVENSION ---
    .rename({col: col.strip().lower().replace(" ", "_") for col in df_scada_dms.columns})
    
    # --- STEP 2: DROP EMPTY OR NULL ROWS---
    # if the ID of the ami_head_end is null, or it is blank or white space, it drops the whole row
    .filter(
        pl.col("system_id").is_not_null() & 
        (pl.col("system_id").str.strip_chars() != "")
    )
    
    # --- STEP 3: INGESTION META DATA  ---
    .with_columns([
        pl.lit(datetime.now()).alias("ingested_at"), 
        pl.lit(SCADA_DMS_FILE).alias("ingested_from_file"),
    ])
    
    # --- STEP 4: TEXT HARMONIZATION ---
    # Not it will only apply "UNKNOWN" to empty valid rows
    .with_columns([
        pl.when(
            pl.col(col).is_null() | (pl.col(col).str.strip_chars() == "")
        )
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col(col).str.strip_chars())
        .alias(col)
        for col in ["provider_id", "function", "control_center"]
    ])
    
    # --- STEP 5: DEDUPLICATION ---
    .unique(maintain_order=True)
)

# 3. Show thrust result
print(cleaned_scada_dms_df)

shape: (10, 6)
┌───────────┬─────────────┬───────────┬───────────────────┬────────────────────────────┬────────────────────┐
│ system_id ┆ provider_id ┆ function  ┆ control_center    ┆ ingested_at                ┆ ingested_from_file │
│ ---       ┆ ---         ┆ ---       ┆ ---               ┆ ---                        ┆ ---                │
│ str       ┆ str         ┆ str       ┆ str               ┆ datetime[μs]               ┆ str                │
╞═══════════╪═════════════╪═══════════╪═══════════════════╪════════════════════════════╪════════════════════╡
│ SCD-001   ┆ UP-002      ┆ SCADA+DMS ┆ Control Center 1  ┆ 2026-08-28 07:51:25.696330 ┆ scada_dms.csv      │
│ SCD-002   ┆ UP-003      ┆ SCADA     ┆ Control Center 2  ┆ 2026-08-28 07:51:25.696330 ┆ scada_dms.csv      │
│ SCD-003   ┆ UP-004      ┆ SCADA+DMS ┆ Control Center 3  ┆ 2026-08-28 07:51:25.696330 ┆ scada_dms.csv      │
│ SCD-004   ┆ UP-003      ┆ SCADA     ┆ Control Center 4  ┆ 2026-08-28 07:51:25.696330 ┆ scada_dms.csv   

## -- DATA CLEANSING & HARMONIZATION   - SMART METERS

In [62]:
# 1 Execution Pipeline
    
# 2. Cleansing with polars
cleaned_smart_meters_df = (
    df_smart_meters
    
    # --- STEP 1: NAMING CONVENSION ---
    .rename({col: col.strip().lower().replace(" ", "_") for col in df_smart_meters.columns})
    
    # --- STEP 2: DROP EMPTY OR NULL ROWS---
    # if the ID of the ami_head_end is null, or it is blank or white space, it drops the whole row
    .filter(
        pl.col("meter_id").is_not_null() & 
        (pl.col("meter_id").str.strip_chars() != "")
    )
    
    # --- STEP 3: INGESTION META DATA  ---
    .with_columns([
        pl.lit(datetime.now()).alias("ingested_at"), 
        pl.lit(SMART_METERS_FILE).alias("ingested_from_file"),
    ])
    
    # --- STEP 4: TEXT HARMONIZATION ---
    # Not it will only apply "UNKNOWN" to empty valid rows
    .with_columns([
        pl.when(
            pl.col(col).is_null() | (pl.col(col).str.strip_chars() == "")
        )
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col(col).str.strip_chars())
        .alias(col)
        for col in ["transformer_id", "consumer_id", "hes_id", "comm_protocol"]
    ])
    
    # --- STEP 5: DEDUPLICATION ---
    .unique(maintain_order=True)
)

# 3. Show thrust result
print(cleaned_smart_meters_df.head(5))

shape: (5, 9)
┌──────────────┬────────────────┬──────────────┬─────────┬──────────────┬──────────────┬───────────────┬─────────────────┬────────────────────┐
│ meter_id     ┆ transformer_id ┆ consumer_id  ┆ hes_id  ┆ install_date ┆ reading_date ┆ comm_protocol ┆ ingested_at     ┆ ingested_from_file │
│ ---          ┆ ---            ┆ ---          ┆ ---     ┆ ---          ┆ ---          ┆ ---           ┆ ---             ┆ ---                │
│ str          ┆ str            ┆ str          ┆ str     ┆ date         ┆ date         ┆ str           ┆ datetime[μs]    ┆ str                │
╞══════════════╪════════════════╪══════════════╪═════════╪══════════════╪══════════════╪═══════════════╪═════════════════╪════════════════════╡
│ MTR-00000001 ┆ DXF-081750     ┆ CON-04550726 ┆ HES-026 ┆ 2019-03-12   ┆ 2026-08-08   ┆ Zigbee        ┆ 2026-08-28      ┆ smart_meters.csv   │
│              ┆                ┆              ┆         ┆              ┆              ┆               ┆ 08:22:48.811204 ┆